### Understanding basic of kernels


we will explore and compare why we need GPUS and what scale we need CPUS and breakdown on kernels optimization as well here 

this blogs is going to be fun how each matrix elements compute with CPU , GPU , GPU + kernels

as we speak GPU process at parallely which is very fast and CPU does it in sequentially 

optimized kernel can improve latency on computing complex matrix multiplication here 

In [ ]:
import gc
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl

architecture_names = {
    (7, 5): "Turing",
    (8, 0): "Ampere (A100-class)",
    (8, 6): "Ampere (consumer-class)",
    (8, 9): "Ada Lovelace",
    (9, 0): "Hopper",
    (10, 0): "Blackwell",
    (12, 0): "Blackwell",
}

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not visible. In VS Code, choose the project's .venv Python kernel, "
        "then verify that `uv run python -c \"import torch; print(torch.cuda.is_available())\"` prints True."
    )

DEVICE_INDEX = torch.cuda.current_device()
DEVICE = torch.device(f"cuda:{DEVICE_INDEX}")
props = torch.cuda.get_device_properties(DEVICE_INDEX)
torch.manual_seed(0)

print(f"PyTorch:             {torch.__version__}")
print(f"PyTorch CUDA build:  {torch.version.cuda}")
print(f"Triton:              {triton.__version__}")
print(f"Device:              {props.name}")
print(f"Compute capability:  {props.major}.{props.minor} (sm_{props.major}{props.minor})")
print(f"CUDA architectures compiled into PyTorch: {torch.cuda.get_arch_list()}")

as you have noticed here that Floast operation has differnet in sizes the more accurate we need 
the more memory we need to store on HBM, and matrix multiplication compute gets heavy you easiy get 
CUDA oom 

In [ ]:
@triton.jit
def matrix_addcmul_kernel(a, b, c, n_cols: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(axis=0)
    column_block = tl.program_id(axis=1)
    columns = column_block * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    offsets = row * n_cols + columns
    mask = columns < n_cols

    # Keep arithmetic consistent while varying the input/output storage dtype.
    a_values = tl.load(a + offsets, mask=mask).to(tl.float32)
    b_values = tl.load(b + offsets, mask=mask).to(tl.float32)
    result = a_values + a_values * b_values
    tl.store(c + offsets, result, mask=mask)

def launch_matrix_addcmul(a, b, c, block_size=256, num_warps=4):
    if not (a.is_cuda and b.is_cuda and c.is_cuda):
        raise ValueError("The Triton path requires CUDA tensors.")
    if not (a.shape == b.shape == c.shape and a.ndim == 2):
        raise ValueError("A, B, and C must be two-dimensional matrices with identical shapes.")
    if not (a.is_contiguous() and b.is_contiguous() and c.is_contiguous()):
        raise ValueError("This learning kernel expects contiguous row-major matrices.")

    rows, cols = a.shape
    grid = (rows, triton.cdiv(cols, block_size))
    matrix_addcmul_kernel[grid](
        a, b, c,
        n_cols=cols, BLOCK_SIZE=block_size, num_warps=num_warps,
    )

MATRIX_ROWS = 10096
MATRIX_COLS = 10096
MATRIX_BLOCK_SIZE = 256
MATRIX_NUM_WARPS = 4
INPUT_SCALE = 0.5
TRITON_WARMUP_MS = 100
TRITON_REP_MS = 300
CPU_WARMUP_RUNS = 3
CPU_REPEATS = 10
ROUND_TRIP_REPEATS = 7

DTYPE_CONFIGS = [
    {
        "label": "FP8 (E4M3)",
        "dtype": getattr(torch, "float8_e4m3fn", None),
        "bits": 8,
        "rtol": 0.125,
        "atol": 0.125,
    },
    {"label": "FP16", "dtype": torch.float16, "bits": 16, "rtol": 1e-3, "atol": 1e-3},
    {"label": "FP32", "dtype": torch.float32, "bits": 32, "rtol": 1e-5, "atol": 1e-6},
]
CPU_PATH = "CPU PyTorch (resident)"
TORCH_GPU_PATH = "PyTorch GPU (resident)"
TRITON_GPU_PATH = "Triton GPU (resident)"
TRITON_ROUND_TRIP_PATH = "Triton GPU + round trip"
BENCHMARK_PATHS = [CPU_PATH, TORCH_GPU_PATH, TRITON_GPU_PATH, TRITON_ROUND_TRIP_PATH]

print(f"Expression: C = A + A * B (elementwise)")
print(f"Shape:      [{MATRIX_ROWS:,}, {MATRIX_COLS:,}]")
print(f"Elements:   {MATRIX_ROWS * MATRIX_COLS:,}")
for config in DTYPE_CONFIGS:
    matrix_mib = MATRIX_ROWS * MATRIX_COLS * config["bits"] / 8 / 2**20
    print(f"One {config['label']:<10} matrix: {matrix_mib:7.1f} MiB")
print(f"CPU worker threads available to PyTorch: {torch.get_num_threads()}")

we will be performing simple add matrix here so we can see compare and see which does it fast and why we choose this architecture instead of this   

In [ ]:
def _latency_stats(samples_ms):
    p20, median, p80 = np.quantile(np.asarray(samples_ms, dtype=np.float64), [0.2, 0.5, 0.8])
    return {"p20_ms": float(p20), "latency_ms": float(median), "p80_ms": float(p80)}


def _cuda_latency_stats(fn, warmup_ms, rep_ms):
    p20, median, p80 = triton.testing.do_bench(
        fn, warmup=warmup_ms, rep=rep_ms, quantiles=[0.2, 0.5, 0.8]
    )
    return {"p20_ms": float(p20), "latency_ms": float(median), "p80_ms": float(p80)}


def _empty_measurement(status, error=None):
    return {
        "p20_ms": np.nan,
        "latency_ms": np.nan,
        "p80_ms": np.nan,
        "status": status,
        "error": error,
    }


def benchmark_matrix_addcmul_dtype(
    source_a,
    source_b,
    fp32_reference,
    dtype_config,
    *,
    block_size=MATRIX_BLOCK_SIZE,
    num_warps=MATRIX_NUM_WARPS,
    triton_warmup_ms=TRITON_WARMUP_MS,
    triton_rep_ms=TRITON_REP_MS,
    cpu_warmup_runs=CPU_WARMUP_RUNS,
    cpu_repeats=CPU_REPEATS,
    round_trip_repeats=ROUND_TRIP_REPEATS,
):
    """Benchmark one storage dtype and return one result row per execution path."""
    label = dtype_config["label"]
    dtype = dtype_config["dtype"]
    bits = dtype_config["bits"]
    numel = source_a.numel()
    matrix_mib = numel * bits / 8 / 2**20
    measurements = {}
    correctness = {
        "correct": False,
        "max_abs_error": np.nan,
        "mean_abs_error": np.nan,
        "max_abs_error_vs_fp32": np.nan,
        "mean_abs_error_vs_fp32": np.nan,
        "peak_gpu_memory_mib": np.nan,
    }

    if dtype is None:
        message = f"{label} is not available in this PyTorch build"
        measurements = {path: _empty_measurement("unavailable", message) for path in BENCHMARK_PATHS}
    else:
        # CPU PyTorch benchmark: inputs and output stay in host memory.
        a_cpu = source_a.to(dtype)
        b_cpu = source_b.to(dtype)

        try:
            c_cpu = torch.empty_like(a_cpu)
            for _ in range(cpu_warmup_runs):
                torch.addcmul(a_cpu, a_cpu, b_cpu, out=c_cpu)
            cpu_samples_ms = []
            for _ in range(cpu_repeats):
                start = time.perf_counter()
                torch.addcmul(a_cpu, a_cpu, b_cpu, out=c_cpu)
                cpu_samples_ms.append((time.perf_counter() - start) * 1e3)
            measurements[CPU_PATH] = {
                **_latency_stats(cpu_samples_ms), "status": "ok", "error": None
            }
        except (RuntimeError, TypeError) as exc:
            measurements[CPU_PATH] = _empty_measurement("unsupported", str(exc))
        finally:
            if "c_cpu" in locals():
                del c_cpu

        # GPU setup: allocate and copy the shared inputs once.
        a_gpu = b_gpu = c_gpu = c_from_gpu = None
        try:
            torch.cuda.reset_peak_memory_stats(DEVICE)
            a_gpu = torch.empty_like(a_cpu, device=DEVICE)
            b_gpu = torch.empty_like(b_cpu, device=DEVICE)
            c_gpu = torch.empty_like(a_gpu)
            c_from_gpu = torch.empty_like(a_cpu)
            a_gpu.copy_(a_cpu)
            b_gpu.copy_(b_cpu)

            # PyTorch GPU benchmark: inputs and output stay on the GPU.
            try:
                torch.addcmul(a_gpu, a_gpu, b_gpu, out=c_gpu)
                torch.cuda.synchronize()
                measurements[TORCH_GPU_PATH] = {
                    **_cuda_latency_stats(
                        lambda: torch.addcmul(a_gpu, a_gpu, b_gpu, out=c_gpu),
                        triton_warmup_ms,
                        triton_rep_ms,
                    ),
                    "status": "ok",
                    "error": None,
                }
            except Exception as exc:
                measurements[TORCH_GPU_PATH] = _empty_measurement("unsupported", str(exc))

            # Triton GPU benchmark: inputs and output stay on the GPU.
            try:
                # The first call compiles the dtype specialization and is not timed.
                launch_matrix_addcmul(a_gpu, b_gpu, c_gpu, block_size, num_warps)
                torch.cuda.synchronize()
                measurements[TRITON_GPU_PATH] = {
                    **_cuda_latency_stats(
                        lambda: launch_matrix_addcmul(a_gpu, b_gpu, c_gpu, block_size, num_warps),
                        triton_warmup_ms,
                        triton_rep_ms,
                    ),
                    "status": "ok",
                    "error": None,
                }

                # Triton round-trip benchmark: CPU → GPU → Triton → CPU.
                round_trip_samples_ms = []
                for _ in range(round_trip_repeats):
                    torch.cuda.synchronize()
                    start = time.perf_counter()
                    a_gpu.copy_(a_cpu)
                    b_gpu.copy_(b_cpu)
                    launch_matrix_addcmul(a_gpu, b_gpu, c_gpu, block_size, num_warps)
                    c_from_gpu.copy_(c_gpu)
                    torch.cuda.synchronize()
                    round_trip_samples_ms.append((time.perf_counter() - start) * 1e3)
                measurements[TRITON_ROUND_TRIP_PATH] = {
                    **_latency_stats(round_trip_samples_ms), "status": "ok", "error": None
                }

                # Correctness check: compare Triton output with storage and FP32 references.
                c_from_gpu.copy_(c_gpu)
                torch.cuda.synchronize()
                actual = c_from_gpu.float()
                quantized_a = a_cpu.float()
                quantized_b = b_cpu.float()
                storage_reference = torch.addcmul(quantized_a, quantized_a, quantized_b).to(dtype).float()
                storage_error = (actual - storage_reference).abs()
                fp32_error = (actual - fp32_reference).abs()
                torch.testing.assert_close(
                    actual, storage_reference, rtol=dtype_config["rtol"], atol=dtype_config["atol"]
                )
                correctness = {
                    "correct": True,
                    "max_abs_error": float(storage_error.max()),
                    "mean_abs_error": float(storage_error.mean()),
                    "max_abs_error_vs_fp32": float(fp32_error.max()),
                    "mean_abs_error_vs_fp32": float(fp32_error.mean()),
                    "peak_gpu_memory_mib": torch.cuda.max_memory_allocated(DEVICE) / 2**20,
                }
            except Exception as exc:
                measurements[TRITON_GPU_PATH] = _empty_measurement("failed", str(exc))
                measurements[TRITON_ROUND_TRIP_PATH] = _empty_measurement("failed", str(exc))
        except Exception as exc:
            for path in (TORCH_GPU_PATH, TRITON_GPU_PATH, TRITON_ROUND_TRIP_PATH):
                measurements.setdefault(path, _empty_measurement("failed", str(exc)))
        finally:
            del a_gpu, b_gpu, c_gpu, c_from_gpu
            del a_cpu, b_cpu
            gc.collect()
            torch.cuda.empty_cache()

    rows = []
    traffic_factors = {CPU_PATH: 3, TORCH_GPU_PATH: 3, TRITON_GPU_PATH: 3, TRITON_ROUND_TRIP_PATH: 6}
    for path in BENCHMARK_PATHS:
        measurement = measurements[path]
        traffic_gb = traffic_factors[path] * numel * bits / 8 / 1e9
        latency_ms = measurement["latency_ms"]
        effective_gbps = traffic_gb / (latency_ms / 1e3) if np.isfinite(latency_ms) else np.nan
        rows.append({
            "dtype": label,
            "dtype_bits": bits,
            "path": path,
            "rows": source_a.shape[0],
            "cols": source_a.shape[1],
            "numel": numel,
            "block_size": block_size,
            "num_warps": num_warps,
            "matrix_mib": matrix_mib,
            "traffic_gb": traffic_gb,
            "effective_gbps": effective_gbps,
            **measurement,
            **correctness,
        })
    return rows


def run_dtype_benchmarks(
    rows=MATRIX_ROWS,
    cols=MATRIX_COLS,
    dtype_configs=DTYPE_CONFIGS,
    *,
    seed=0,
    input_scale=INPUT_SCALE,
    **benchmark_kwargs,
):
    """Run the same data through every dtype and return a tidy DataFrame."""
    generator = torch.Generator(device="cpu").manual_seed(seed)
    source_a = torch.randn((rows, cols), dtype=torch.float32, generator=generator).mul_(input_scale)
    source_b = torch.randn((rows, cols), dtype=torch.float32, generator=generator).mul_(input_scale)
    fp32_reference = torch.addcmul(source_a, source_a, source_b)

    result_rows = []
    try:
        for dtype_config in dtype_configs:
            print(f"Benchmarking {dtype_config['label']} ...")
            result_rows.extend(benchmark_matrix_addcmul_dtype(
                source_a, source_b, fp32_reference, dtype_config, **benchmark_kwargs
            ))
    finally:
        del source_a, source_b, fp32_reference
        gc.collect()

    results = pd.DataFrame(result_rows)
    fp32_latency = (
        results.loc[results["dtype"].eq("FP32"), ["path", "latency_ms"]]
        .rename(columns={"latency_ms": "fp32_latency_ms"})
    )
    results = results.merge(fp32_latency, on="path", how="left")
    results["speedup_vs_fp32"] = results["fp32_latency_ms"] / results["latency_ms"]
    torch_gpu_latency = (
        results.loc[results["path"].eq(TORCH_GPU_PATH), ["dtype", "latency_ms"]]
        .rename(columns={"latency_ms": "torch_gpu_latency_ms"})
    )
    results = results.merge(torch_gpu_latency, on="dtype", how="left")
    results["speedup_vs_torch_gpu"] = results["torch_gpu_latency_ms"] / results["latency_ms"]
    return results

In [ ]:
benchmark_results = run_dtype_benchmarks()

In [ ]:
def build_gpu_comparison_summary(results):
    """Return one easy-to-scan comparison row per dtype."""
    dtype_order = [config["label"] for config in DTYPE_CONFIGS]
    latency = results.pivot(index="dtype", columns="path", values="latency_ms")
    status = results.pivot(index="dtype", columns="path", values="status")
    triton_rows = results.loc[results["path"].eq(TRITON_GPU_PATH)].set_index("dtype")

    summary = pd.DataFrame(index=dtype_order)
    summary.index.name = "dtype"
    summary["CPU PyTorch (ms)"] = latency.reindex(dtype_order).get(CPU_PATH)
    summary["PyTorch GPU (ms)"] = latency.reindex(dtype_order).get(TORCH_GPU_PATH)
    summary["Triton GPU (ms)"] = latency.reindex(dtype_order).get(TRITON_GPU_PATH)
    summary["Triton speedup vs CPU"] = (
        summary["CPU PyTorch (ms)"] / summary["Triton GPU (ms)"]
    )
    summary["Triton speedup vs PyTorch"] = (
        summary["PyTorch GPU (ms)"] / summary["Triton GPU (ms)"]
    )
    summary["Round trip (ms)"] = latency.reindex(dtype_order).get(TRITON_ROUND_TRIP_PATH)
    summary["Transfer penalty"] = summary["Round trip (ms)"] / summary["Triton GPU (ms)"]
    summary["Max error vs FP32"] = triton_rows.reindex(dtype_order)["max_abs_error_vs_fp32"]
    summary["CPU status"] = status.reindex(dtype_order).get(CPU_PATH)
    summary["PyTorch status"] = status.reindex(dtype_order).get(TORCH_GPU_PATH)
    summary["Triton status"] = status.reindex(dtype_order).get(TRITON_GPU_PATH)
    return summary.reset_index()


gpu_comparison_summary = build_gpu_comparison_summary(benchmark_results)
display(gpu_comparison_summary.round(4))

In [ ]:
DTYPE_ORDER = ["FP8 (E4M3)", "FP16", "FP32"]

PLOT_PATHS = [
    CPU_PATH,
    TORCH_GPU_PATH,
    TRITON_GPU_PATH,
    TRITON_ROUND_TRIP_PATH,
]

PATH_LABELS = {
    CPU_PATH: "CPU PyTorch",
    TORCH_GPU_PATH: "PyTorch GPU",
    TRITON_GPU_PATH: "Triton GPU",
    TRITON_ROUND_TRIP_PATH: "Triton + transfers",
}

PATH_COLORS = {
    CPU_PATH: "#6B7280",
    TORCH_GPU_PATH: "#4C78A8",
    TRITON_GPU_PATH: "#F58518",
    TRITON_ROUND_TRIP_PATH: "#B279A2",
}


def _path_plot_values(results, path, dtype_order):
    """Return plot-ready latency data without modifying results."""
    path_rows = (
        results.loc[results["path"].eq(path)]
        .drop_duplicates("dtype", keep="last")
        .set_index("dtype")
    )

    latency = np.full(len(dtype_order), np.nan, dtype=np.float64)
    p20 = np.full(len(dtype_order), np.nan, dtype=np.float64)
    p80 = np.full(len(dtype_order), np.nan, dtype=np.float64)
    available = np.zeros(len(dtype_order), dtype=bool)

    for index, dtype in enumerate(dtype_order):
        if dtype not in path_rows.index:
            continue

        row = path_rows.loc[dtype]
        status = str(row["status"]).lower()
        median = float(row["latency_ms"])

        if status != "ok" or not np.isfinite(median):
            continue

        latency[index] = median
        p20[index] = float(row["p20_ms"])
        p80[index] = float(row["p80_ms"])
        available[index] = True

    lower_error = np.where(np.isfinite(p20), np.maximum(latency - p20, 0.0), 0.0)
    upper_error = np.where(np.isfinite(p80), np.maximum(p80 - latency, 0.0), 0.0)
    return latency, np.vstack((lower_error, upper_error)), available


def plot_addcmul_latency_comparison(results, *, log_scale=True):
    """Plot all execution-path latencies and Triton transfer overhead."""
    dtype_order = [dtype for dtype in DTYPE_ORDER if dtype in set(results["dtype"].dropna())]
    x = np.arange(len(dtype_order), dtype=np.float64)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    latency_ax, transfer_ax = axes

    # Panel 1: directly compare every available CPU and GPU execution path.
    width = 0.80 / len(PLOT_PATHS)
    plotted_latencies = []

    for path_index, path in enumerate(PLOT_PATHS):
        latency, error, available = _path_plot_values(results, path, dtype_order)
        valid = available & np.isfinite(latency)
        if not valid.any():
            continue

        positions = x + (path_index - (len(PLOT_PATHS) - 1) / 2) * width
        bars = latency_ax.bar(
            positions[valid],
            latency[valid],
            width=width * 0.92,
            yerr=error[:, valid],
            capsize=3,
            color=PATH_COLORS[path],
            label=PATH_LABELS[path],
            edgecolor="white",
            linewidth=0.7,
        )
        latency_ax.bar_label(
            bars,
            labels=[f"{value:.3f} ms" for value in latency[valid]],
            padding=3,
            fontsize=9,
            rotation=90,
        )
        plotted_latencies.extend(latency[valid])

    latency_ax.set_title("Latency comparison", fontsize=13, fontweight="bold")
    latency_ax.set_ylabel("Median latency (ms)", fontsize=11)
    latency_ax.set_xticks(x, dtype_order, fontsize=10)
    latency_ax.grid(axis="y", linestyle="--", alpha=0.3)
    latency_ax.set_axisbelow(True)
    latency_ax.legend(frameon=False, fontsize=9, ncol=2, loc="upper left")

    if log_scale:
        latency_ax.set_yscale("log")
    if plotted_latencies:
        positive = np.asarray(plotted_latencies, dtype=np.float64)
        positive = positive[positive > 0]
        if positive.size:
            if log_scale:
                latency_ax.set_ylim(positive.min() * 0.65, positive.max() * 2.4)
            else:
                latency_ax.set_ylim(0, positive.max() * 1.25)

    fp8_torch_latency, _, fp8_torch_available = _path_plot_values(
        results, TORCH_GPU_PATH, ["FP8 (E4M3)"]
    )
    if (
        "FP8 (E4M3)" in dtype_order
        and (not fp8_torch_available[0] or not np.isfinite(fp8_torch_latency[0]))
    ):
        latency_ax.text(
            0.0,
            -0.20,
            "Note: FP8 PyTorch GPU is unavailable for torch.addcmul.",
            transform=latency_ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            color="#666666",
        )

    # Panel 2: show how much CPU↔GPU transfers inflate Triton latency.
    resident_latency, _, resident_available = _path_plot_values(
        results, TRITON_GPU_PATH, dtype_order
    )
    round_trip_latency, _, round_trip_available = _path_plot_values(
        results, TRITON_ROUND_TRIP_PATH, dtype_order
    )
    transfer_ratio = np.full(len(dtype_order), np.nan, dtype=np.float64)
    np.divide(
        round_trip_latency,
        resident_latency,
        out=transfer_ratio,
        where=resident_available & round_trip_available & (resident_latency > 0),
    )
    valid_ratio = (
        resident_available
        & round_trip_available
        & np.isfinite(transfer_ratio)
        & (resident_latency > 0)
    )

    ratio_bars = transfer_ax.bar(
        x[valid_ratio],
        transfer_ratio[valid_ratio],
        width=0.58,
        color=PATH_COLORS[TRITON_ROUND_TRIP_PATH],
        edgecolor="white",
        linewidth=0.7,
    )
    transfer_ax.bar_label(
        ratio_bars,
        labels=[f"{value:.1f}× slower" for value in transfer_ratio[valid_ratio]],
        padding=4,
        fontsize=10,
        fontweight="bold",
    )
    transfer_ax.axhline(1.0, color="#555555", linestyle="--", linewidth=1.1)
    transfer_ax.set_title("Transfer overhead", fontsize=13, fontweight="bold")
    transfer_ax.set_ylabel("Round-trip / resident Triton latency", fontsize=11)
    transfer_ax.set_xticks(x, dtype_order, fontsize=10)
    transfer_ax.grid(axis="y", linestyle="--", alpha=0.3)
    transfer_ax.set_axisbelow(True)

    if valid_ratio.any():
        transfer_ax.set_ylim(0, transfer_ratio[valid_ratio].max() * 1.22)

    fig.suptitle("addcmul latency comparison", fontsize=16, fontweight="bold", y=0.99)
    fig.text(
        0.5,
        0.925,
        "C = A + A × B | bars show median latency, whiskers show p20–p80",
        ha="center",
        fontsize=10,
        color="#444444",
    )

    if "props" in globals() and not results.empty:
        config = results.iloc[0]
        fig.text(
            0.5,
            0.025,
            (
                f"GPU: {props.name} | sm_{props.major}{props.minor} | "
                f"shape: {int(config.rows):,} × {int(config.cols):,} | "
                f"block: {int(config.block_size)} | warps: {int(config.num_warps)}"
            ),
            ha="center",
            fontsize=9,
            color="#555555",
        )

    plt.tight_layout(rect=(0.03, 0.12, 0.99, 0.87), w_pad=3.0)
    return fig, axes

In [ ]:
benchmark_figure, benchmark_axes = plot_addcmul_latency_comparison(benchmark_results)
plt.show()